# Домашнее задание 2

**Дисциплина** Алгоритмы и структуры данных

**Тема** Алгоритмы на графах

# Задание 1. Двоичное дерево поиска

| Ограничение | Значение |
| ----------- | -------- |
| Ограничение времени | 1 секунда |
| Ограничение памяти | 64 Мб |
| Ввод | input.txt |
| Вывод | output.txt |

Дано двоичное дерево. Требуется определить, является ли заданное дерево двоичным деревом поиска.

Для дерева поиска характерна упорядоченность: для каждого узла ВСЕ значения в левом поддереве меньше значения узла, а в правом поддереве — больше.

**Формат ввода**

В первой строке записано число вершин $n$

Вершины дерева нумеруются числами от $0$ до $n-1$.

Известно, что вершина $0$ является корнем.

В каждой из следующих $n$ строк дана информация об узлах $0, 1, ..., n-1$ в следующем виде:
- для $i$‑ого узла известно $value_i$ - его значение,  
- $left_i$ и $right_i$ - индексы левого и правого потомков.

Если у вершины $i$ нет одного из потомков или нет обоих, соответствующее значение равно $-1$.

**Формат вывода**

Если дерево является двоичным деревом поиска, выведите TRUE, иначе, выведите FALSE.

In [1]:
data = '''3
2 1 2
1 -1 -1
3 -1 -1
'''

with open('input.txt', 'w', encoding='utf-8') as f:
    f.write(data)

In [2]:
import sys
sys.setrecursionlimit(10**7)

def is_bst(tree):
    def check_bst(node, min_val, max_val):
        if node == -1:
            return True
        val, left, right = tree[node]
        if val <= min_val or val >= max_val:
            return False
        return check_bst(left, min_val, val) and check_bst(right, val, max_val)

    return check_bst(0, float('-inf'), float('inf')) if tree else True

with open('input.txt', 'r', encoding='utf-8') as fin:
    n = int(fin.readline().strip())
    tree = []
    for _ in range(n):
        value, left, right = map(int, fin.readline().split())
        tree.append((value, left, right))

result = is_bst(tree)

with open('output.txt', 'w', encoding='utf-8') as fout:
    fout.write("TRUE\n" if result else "FALSE\n")

# Задание 2. Поиск пути

| Ограничение | Значение |
| ----------- | -------- |
| Ограничение времени | 1 секунда |
| Ограничение памяти | 64 Мб |
| Ввод | input.txt |
| Вывод | output.txt |

Компания "Быстрый и экономный" осуществляет перевозки грузов между $n$ городами. Города соединены сетью дорог, где каждая дорога имеет два ключевых параметра:
- Время проезда (в часах) - определяет длительность перевозки
- Стоимость проезда (в денежных единицах) - включает платные участки, топливо и другие расходы

Компания получила срочный заказ на доставку ценного груза из города $s$ в город $t$, но у компании есть ограничения на бюджет: она может выделить только $C$ денежных единиц на транспортные расходы для этой доставки.

Задача: найти маршрут доставки, который удовлетворяет бюджетному ограничению и при этом минимизирует общее время доставки.

*Учтите, что:*
- Дороги могут быть односторонними (ориентированный граф)
- Время и стоимость проезда по каждой дороге известны заранее
- Компания может использовать любые доступные маршруты, включая проезд через промежуточные города

*Формальная постановка задачи:*

Дан ориентированный граф, где каждое ребро имеет длину и стоимость. Найдите кратчайший путь из $s$ в $t$ где суммарная стоимость пути не превышает $C$.

**Формат ввода**

Первая строка: $n$, $m$, $s$, $t$, $C$
где
- $n$ - количество городов (1 ≤ $n$ ≤ 1000)
- $m$ - количество дорог (0 ≤ $m$ ≤ 10000)
- $s$ - город отправления (1 ≤ $s$ ≤ n)
- $t$ - город назначения (1 ≤ $t$ ≤ n)
- $C$ - максимальный бюджет (1 ≤ $C$ ≤ 10^6)

Следующие $m$ строк имеют формат: $u$, $v$, $len$, $cost$
- $u$, $v$ - города, соединенные дорогой (от $u$ к $v$)
- $len$ - время проезда в часах
- $cost$ - стоимость проезда

**Формат вывода**

Требуется вывести минимальное время доставки (суммарное время проезда по выбранному маршруту) или, если доставка в рамках бюджета невозможна, вывести -1.

**Примечания**

Компания должна доставить оборудование из города 1 в город 4 с бюджетом 10 единиц. Доступны дороги:

1->2: 2 часа, стоимость 3

2->3: 3 часа, стоимость 4

3->4: 1 час, стоимость 2

1->3: 6 часов, стоимость 5

1->4: 10 часов, стоимость 8

2->4: 3 часов, стоимость 10

Оптимальный маршрут: 1->2->3->4 со временем 6 часов и стоимостью 9 (в рамках бюджета)

In [3]:
data = '''4 4 1 4 10
1 2 2 3
2 3 3 4
3 4 1 2
1 3 6 5
'''

with open('input.txt', 'w', encoding='utf-8') as f:
    f.write(data)

In [4]:
import heapq

def find_min_time(n, m, s, t, C, edges):
    graph = [[] for _ in range(n+1)]
    for u, v, length, cost in edges:
        graph[u].append((v, length, cost))

    dist = [[float('inf')] * (C+1) for _ in range(n+1)]
    dist[s][0] = 0

    pq = [(0, s, 0)]

    while pq:
        time, node, cost = heapq.heappop(pq)
        if node == t:
            return time
        if dist[node][cost] < time:
            continue
        for nxt, length, nxt_cost in graph[node]:
            new_cost = cost + nxt_cost
            if new_cost <= C and dist[nxt][new_cost] > time + length:
                dist[nxt][new_cost] = time + length
                heapq.heappush(pq, (time + length, nxt, new_cost))

    return -1

with open('input.txt', 'r', encoding='utf-8') as fin:
    n, m, s, t, C = map(int, fin.readline().split())
    edges = [tuple(map(int, fin.readline().split())) for _ in range(m)]

result = find_min_time(n, m, s, t, C, edges)

with open('output.txt', 'w', encoding='utf-8') as fout:
    fout.write(str(result) + '\n')